# 그림체 검색기 놀이터 — 15분 안에 첫 점수 내기

DLthon 그림체 RAG 팀입니다. 이 노트북 하나로 **환경 세팅부터 리더보드 등록까지** 됩니다.

**하는 일 한 줄**: 그림 한 장을 던지면 **같은 그림체의 그림**을 찾아오는 검색기를 만듭니다.
잘 찾아오는지는 자동으로 채점됩니다.

**당신이 쓸 코드는 함수 하나가 전부입니다.**

```python
def encode(images):        # 그림 리스트를 받아서
    return vectors         # 숫자 배열 (N, D) 로 바꿔주면 끝
```

가까운 숫자끼리 같은 그림체이면 점수가 높습니다. 그게 전부입니다.

---

## 시작 전에 딱 하나 — 드라이브 바로가기 추가

공유받은 폴더는 **그냥은 코랩에 안 보입니다.** 한 번만 해두면 됩니다.

1. 구글 드라이브에서 공유받은 **`DLthon_그림체RAG`** 폴더를 찾습니다 (공유 문서함에 있습니다)
2. 폴더에 **우클릭 → "내 드라이브에 바로가기 추가"**
3. 위치는 **내 드라이브 최상위**로 두세요

이걸 안 하면 아래 첫 셀에서 "폴더를 못 찾겠다"고 나옵니다.

**(선택) GPU 를 켜면 빠릅니다**: 상단 메뉴 → 런타임 → 런타임 유형 변경 → GPU (T4)
CPU 로도 됩니다. 조금 느릴 뿐입니다.

> 채점 부분(3·4·5·6절)은 제 컴퓨터에서 **압축 파일 그대로 풀어 끝까지 돌려 확인했습니다.**
> 다만 **코랩에서는 아직 아무도 안 돌려봤습니다.** 뭔가 어긋나면 바로 팀 채널에 알려주세요 — 제가 고칩니다.

## 1. 세팅 (5분)

In [ ]:
# 드라이브 연결
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob, sys, subprocess

# 공유 폴더 찾기 (바로가기 위치가 사람마다 조금 다를 수 있어서 몇 군데 뒤진다)
# 바로가기를 어디에 뒀든 찾는다. 마지막 경로가 핵심 —
# 공유 폴더 바로가기는 실제로 .shortcut-targets-by-id 아래에 풀린다
CANDS = glob.glob('/content/drive/MyDrive/DLthon_그림체RAG') \
      + glob.glob('/content/drive/MyDrive/*/DLthon_그림체RAG') \
      + glob.glob('/content/drive/MyDrive/*/*/DLthon_그림체RAG') \
      + glob.glob('/content/drive/Shareddrives/*/DLthon_그림체RAG') \
      + glob.glob('/content/drive/.shortcut-targets-by-id/*/DLthon_그림체RAG')

if not CANDS:
    raise SystemExit(
        "공유 폴더를 못 찾았습니다.\n"
        "  드라이브에서 'DLthon_그림체RAG' 폴더 우클릭 -> '내 드라이브에 바로가기 추가' 를 하고\n"
        "  이 셀을 다시 실행해 주세요. (공유 문서함에 있는 건 코랩에서 안 보입니다)")

SHARE = CANDS[0]
print("공유 폴더:", SHARE)
print("들어 있는 것:", sorted(os.path.basename(p) for p in glob.glob(SHARE + '/*')))

In [ ]:
# 데이터와 도구를 코랩 로컬로 푼다 (드라이브에서 바로 읽으면 느리다)
import zipfile, time
t0 = time.time()

for z, dst in [('daypack_v2.zip', '/content'), ('kit.zip', '/content')]:
    src = os.path.join(SHARE, z)
    if not os.path.exists(src):
        raise SystemExit(f"{z} 가 공유 폴더에 없습니다. 팀장에게 알려주세요.")
    with zipfile.ZipFile(src) as f:
        f.extractall(dst)
    print(f"  {z} 풀었습니다")

PACK = '/content/daypack_v2'
KIT  = '/content/kit'
n = len(glob.glob(PACK + '/images/*/*.jpg'))
print(f"\n그림 {n}장 / {len(glob.glob(PACK + '/images/*'))}클래스  ({time.time()-t0:.0f}초)")

In [ ]:
# 패키지 (코랩엔 torch·torchvision 이 이미 있다. transformers 만 확인)
!pip -q install "transformers>=4.40,<5" 2>&1 | tail -1
print("준비 끝")

### (선택) 바로가기 추가가 잘 안 되면 — 폴더 ID 로 직접 받기

위 셀이 계속 폴더를 못 찾으면 이걸 대신 쓰세요. **바로가기 없이** 공유 폴더에서 직접 내려받습니다.
인증 팝업이 한 번 뜨니 승인해 주세요.

> 팀장 메모: 이 방법은 제가 코랩에서 직접 확인하지 못했습니다.
> 안 되면 위쪽 "바로가기 추가" 쪽을 쓰시고 팀 채널에 알려주세요.

In [ ]:
# 방법 B — 바로가기 없이 폴더 ID 로 받기 (위 셀이 실패했을 때만 실행)
FOLDER_ID = '1QWjKusxrk8G8p57KShZSGCVb16pqd9Qm'

from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io, os

svc = build('drive', 'v3')
res = svc.files().list(
    q=f"'{FOLDER_ID}' in parents and trashed=false",
    fields="files(id,name,size)", pageSize=100).execute()
files = {f['name']: f['id'] for f in res.get('files', [])}
print("폴더에 있는 것:", sorted(files))

SHARE = '/content/share'
os.makedirs(SHARE, exist_ok=True)

for name in ['daypack_v2.zip', 'kit.zip']:
    if name not in files:
        raise SystemExit(f"{name} 이 폴더에 없습니다. 팀장에게 알려주세요.")
    dst = os.path.join(SHARE, name)
    with io.FileIO(dst, 'wb') as fh:
        dl = MediaIoBaseDownload(fh, svc.files().get_media(fileId=files[name]))
        done = False
        while not done:
            status, done = dl.next_chunk()
            if status:
                print(f"  {name} {int(status.progress() * 100)}%", end='\r')
    print(f"  {name} 받았습니다 ({os.path.getsize(dst) / 1e6:.0f}MB)")

print("\n이제 위의 '압축 풀기' 셀을 실행하세요.")

## 2. 먼저 판을 눈으로 봅니다

**숫자를 보기 전에 그림을 봅니다.** 뭘 구분하려는 건지 눈으로 알아야 방법이 떠오릅니다.

세 종류가 섞여 있습니다.

| 이름 | 무엇 | 난이도 |
|---|---|---|
| `ink_m1` ~ `ink_m9` | 흑백 회화 기법 9종 (연필선 ~ 짙은 먹) | **어렵다** — 색 단서가 없습니다 |
| `paint_*` | 서양화 사조 6종 (인상주의, 입체파 …) | 중간 |
| `vec_*` | 벡터 일러스트 5종 (이모지, unDraw …) | 쉽다 |

일부러 쉬운 것도 넣었습니다. 실제 서비스는 전 코퍼스에서 찾아야 하니까요.
대신 채점은 **클래스별·도메인별로 따로** 나옵니다.

In [ ]:
import csv, collections, random
import matplotlib.pyplot as plt
from PIL import Image

rows = list(csv.DictReader(open(f'{PACK}/meta.csv', encoding='utf-8')))
by = collections.defaultdict(list)
for r in rows:
    by[r['style']].append(r['file'])

random.seed(0)
ks = sorted(by)
fig, axes = plt.subplots(len(ks), 5, figsize=(11, 2.0 * len(ks)))
for i, k in enumerate(ks):
    for j, f in enumerate(random.sample(by[k], 5)):
        ax = axes[i][j]
        ax.imshow(Image.open(f'{PACK}/{f}'))
        ax.axis('off')
        if j == 0:
            ax.set_title(f'{k}  (n={len(by[k])})', fontsize=9, loc='left')
plt.tight_layout(); plt.show()

### 잠깐 — 여기서 감을 잡고 갑니다

`ink_m4` ~ `ink_m8` 을 비교해 보세요. **사람 눈에도 애매합니다.** 그게 이 판이 어려운 이유입니다.
반대로 `vec_*` 는 한눈에 갈립니다.

그리고 하나 더. 같은 `ink` 안에는 **똑같은 그림을 다른 기법으로 그린 짝**이 들어 있습니다.
이게 함정입니다 — 검색기가 "그림체가 같은 것"이 아니라 **"내용이 같은 것"** 을 찾아와도 점수가 오릅니다.

실측으로 그 차이가 이렇습니다.

| 가장 가까운 5장에 오는 것 | 무작위 대비 |
|---|---|
| 같은 **내용** (같은 그림, 다른 기법) | **102배** |
| 같은 **그림체** (다른 그림, 같은 기법) | **1.5배** |

즉 그냥 재면 그림체가 아니라 내용을 재게 됩니다.
**그래서 채점기가 "자기 자신"과 "같은 원본/같은 작가"를 이웃 후보에서 빼고 잽니다.** 신경 안 쓰셔도 됩니다.

## 3. 기준선 돌려서 첫 점수 (3분)

먼저 **CLIP 임베딩을 그대로** 쓴 것입니다. 이게 출발점이고, 여러분이 이겨야 할 선입니다.

In [ ]:
%cd /content/kit
!DAYPACK=/content/daypack_v2 python score.py encoders/clip_base.py --name "CLIP 기준선(내 실행)"

### 점수 읽는 법 — 세 가지만

**(1) `p@5`** = 가까운 5장 중 같은 그림체가 몇 장인지의 비율.
`0.40` 이면 5장 중 2장이 맞은 것입니다.

**(2) `무작위`와 `배수`** — 클래스마다 장수가 달라서, 아무거나 집어도 맞는 몫이 있습니다.
그걸로 나눈 게 `배수`입니다. **원점수보다 배수를 보세요.**

**(3) `95% 구간`** — 같은 코드를 두 번 돌려도 숫자는 흔들립니다. 그 흔들림 폭입니다.
**이 폭보다 작은 차이는 "이긴 것"이 아닙니다.**

그리고 맨 아래 **도메인 표**를 꼭 보세요. `전체판` 이 높고 `도메인안` 이 낮으면,
"수묵화냐 이모지냐" 같은 **공짜 문제로 점수를 번 것**입니다.

## 4. 두 번째 기준선 — Gram (선택, 2~3분 더)

**내용을 버리고 "어떻게 칠했나"만 남기는** 방법입니다.
VGG 중간층에서 채널끼리 내적하면 위치 정보(H, W)가 사라지고 붓결·질감만 남습니다.

지금 리더보드 1등이지만 **약점이 있습니다.** 돌려보고 도메인 표를 비교해 보세요.

In [ ]:
!DAYPACK=/content/daypack_v2 python score.py encoders/gram_vgg19.py --name "Gram VGG19(내 실행)"

### ★여기가 아직 안 풀린 지점입니다

두 기준선의 도메인 표를 비교해 보면 이렇습니다.

| 도메인 | CLIP | Gram | 승자 |
|---|---|---|---|
| ink | 0.2262 | **0.3595** | Gram |
| **paint** | **0.5324** | 0.3706 | **CLIP** |
| vector | 0.6510 | **0.8481** | Gram |

**회화(paint)에서는 Gram 이 무너집니다.** `paint_Cubism` 은 CLIP 0.70 vs Gram 0.28.

짐작은 있습니다 — 사조는 붓결보다 **색과 구도**로 갈리는데 Gram 은 그걸 버립니다.
하지만 **확인은 안 했습니다.** 그리고 짝지어 비교하면 Gram 이 이겼어도
**606개 질의에선 CLIP 이 나았습니다.**

그래서 **"최고의 표현 하나"를 찾는 문제가 아닐 수도 있습니다.** 여기가 우리가 풀 자리입니다.

## 5. ★여기부터 당신 차례입니다

아래는 **아주 단순한 예시**입니다 — 그림의 색 분포(히스토그램)만 봅니다. 딱 12줄입니다.

**★그런데 이게 생각보다 잘 나옵니다.** 제가 미리 돌려본 결과입니다.

| | 전체 | ink | paint | vector |
|---|---|---|---|---|
| CLIP 기준선 | **0.3986** | 0.2262 | **0.5324** | 0.6510 |
| 색 히스토그램 (아래 12줄) | 0.3656 | **0.2399** | 0.2733 | **0.8086** |

전체로는 CLIP 에 지는데(짝지어 비교 -0.0330), **ink 에서는 CLIP 을 이깁니다.** vector 는 크게 이깁니다.

왜 그럴까요? **ink 는 흑백입니다.** 그러니 "색 히스토그램"이 사실상 **먹의 농담(톤) 분포**가 됩니다.
짙은 먹과 옅은 선묘는 밝기 분포만으로도 갈립니다. 반대로 회화(paint)에서는 크게 집니다.

**여기서 힌트 하나**: 무거운 모델이 항상 이기는 게 아닙니다. **어떤 신호를 보느냐**가 도메인마다 다릅니다.
12줄짜리 코드가 특정 도메인에서 CLIP 을 이긴다는 게 이 프로젝트가 열려 있다는 증거입니다.

**이걸 고치는 게 시작입니다.**

규칙은 두 개뿐입니다.
- 함수 이름과 모양을 지킨다: `encode(images) -> (N, D) 배열`
- **정답표(`meta.csv`)를 학습에 쓰지 않는다** (그건 답을 보고 푸는 것)

나머지는 자유입니다. 학습을 시켜도 되고 안 시켜도 됩니다.

In [ ]:
%%writefile /content/kit/encoders/my_method.py
# 내 방법 — 색 히스토그램만 보는 아주 단순한 예시
# 이걸 고쳐서 점수를 올리는 게 목표다
import numpy as np

def encode(images):
    out = []
    for im in images:
        a = np.asarray(im.convert('RGB'), dtype='float32') / 255.0
        v = []
        for c in range(3):                       # R, G, B 각각 32칸 히스토그램
            h, _ = np.histogram(a[:, :, c], bins=32, range=(0, 1))
            v.append(h / (h.sum() + 1e-8))
        out.append(np.concatenate(v))
    return np.asarray(out, dtype='float32')

In [ ]:
# 이름을 자기 것으로 바꿔서 돌리세요. 리더보드에 그대로 올라갑니다
# (예시 그대로 돌리면 전체 0.3656 / ink 0.2399 / paint 0.2733 / vector 0.8086 이 나옵니다.
#  숫자가 다르면 뭔가 어긋난 것이니 팀 채널에 알려주세요)
!DAYPACK=/content/daypack_v2 python score.py encoders/my_method.py \
    --name "홍길동-색히스토그램" --note "예시 그대로 돌려본 것"

### 다음에 뭘 해볼까 (힌트)

- **Gram 을 저층만** 쓰기 (`gram_vgg19.py` 의 `LAYERS` 를 `{1: ..., 6: ...}` 로) — 차원이 17만에서 1만으로 줍니다
- **흑백으로 바꿔서** 재기 — 색을 지우면 붓결만 남습니다. `ink` 에 유리할까요 불리할까요?
- **CLIP 과 Gram 을 이어 붙이기**(concat) — 위 표를 보면 이게 제일 유망합니다. 서로 이기는 데가 다릅니다
- **CLIP 에서 내용 성분을 빼기** — 같은 내용 그림들의 평균 방향을 구해 그걸 지우면?
- 아예 다른 표현: 엣지 통계, 주파수(FFT), 국소 이진 패턴(LBP) …

**아무거나 꽂아도 됩니다. 그게 이 놀이터의 요점입니다.**

## 6. 이겼는지 제대로 판정하기

평균이 올랐다고 이긴 게 아닙니다. **같은 그림들로 채점받았으니 질의별로 짝지어 비교**하면
그림 난이도 때문에 생기는 출렁임이 상쇄되고 방법의 차이만 남습니다.

In [ ]:
!python compare.py "CLIP 기준선(내 실행)" "홍길동-색히스토그램"

**차이의 95% 구간이 0을 품으면 "아직 모른다"** 입니다.
그리고 승패 줄을 보세요 — 평균이 낮아도 **어떤 질의에선 이기고 있을 수 있습니다.** 거기에 힌트가 있습니다.

## 7. 결과 공유 — 실행하면 자동으로 모입니다

아래 셀을 돌리면 결과가 **공유 폴더 `results/`** 에 저장됩니다.
따로 보고서를 쓰지 않아도 팀 리더보드가 합쳐집니다.

In [ ]:
import shutil, glob, os
MY_NAME = "홍길동"          # ← 자기 이름으로 바꾸세요

dst = os.path.join(SHARE, 'results')
os.makedirs(dst, exist_ok=True)
sent = 0
for f in glob.glob('/content/kit/runs/*.json'):
    shutil.copy2(f, os.path.join(dst, f"{MY_NAME}_{os.path.basename(f)}"))
    sent += 1
shutil.copy2('/content/kit/leaderboard.csv', os.path.join(dst, f"{MY_NAME}_leaderboard.csv"))
print(f"{sent}개 결과 + 리더보드를 공유 폴더에 올렸습니다 -> {dst}")
print("(폴더에 쓰기 권한이 없다고 나오면 팀장에게 '편집자' 권한을 요청하세요)")

---

## 막히면

- **첫 셀에서 폴더를 못 찾는다** → 드라이브에서 공유 폴더 우클릭 → "내 드라이브에 바로가기 추가"
- **느리다** → 런타임 → 런타임 유형 변경 → GPU (T4)
- **Gram 이 메모리 부족** → `gram_vgg19.py` 의 `LAYERS` 를 `{1: 'relu1_1', 6: 'relu2_1'}` 로 줄이세요
- **그 외** → 팀 채널에 물어보세요. 혼자 30분 넘게 붙잡지 마세요

## 알아두면 좋은 것 — 팀장이 준비하면서 틀린 것 세 개

셋 다 **에러가 안 났고 숫자도 그럴듯했습니다.** 숫자를 의심하는 습관이 이 프로젝트의 핵심입니다.

1. **작가를 맞히고 있었다** — "그림체가 갈린다 5.3배"가 사실은 같은 화가를 찾은 것.
   작가당 8장 상한 + 질의 작가 제외로 고치니 2.9배 (채점기에 반영돼 있습니다)
2. **내용을 재고 있었다** — 내용 102배 vs 그림체 1.5배 (위 2절)
3. **수집 스크립트가 250장 받고 전부 틀린 그림이었다** — 12장을 눈으로 보고 잡았습니다

데이터 출처와 이용 조건은 공유 폴더의 **`팀_공유_브리핑.md`** 9절에 정리돼 있습니다.
**ink(AI-Hub)와 이라스토야 이미지는 공개 저장소·공개 링크에 올리지 말아 주세요.**